# Optimisation d Hyperparametres avec Optuna

## Contexte et Objectifs

Ce notebook est un guide pratique pour l'optimisation d'hyperparametres a l'aide de la bibliotheque `Optuna`. Trouver la bonne combinaison d'hyperparametres est l'une des etapes les plus critiques pour maximiser la performance d'un modele de machine learning. Optuna automatise ce processus de maniere efficace et intelligente.

### Approche de Qualite Industrielle :

1.  **Definition d'une Fonction Objective :** Nous definissons une fonction objective qui encapsule l'entrainement et l'evaluation d'un modele. Optuna se chargera d'appeler cette fonction avec differentes combinaisons d'hyperparametres.
2.  **Espace de Recherche Intelligent :** Optuna permet de definir un espace de recherche pour chaque hyperparametre (par exemple, un intervalle pour le `learning_rate`, des choix specifiques pour le `max_depth`).
3.  **Algorithmes d'Echantillonnage et d'Elagage :** Optuna utilise des algorithmes avances (par exemple, TPE) pour explorer l'espace de recherche de maniere plus intelligente qu'une recherche aleatoire ou en grille. Il peut egalement arreter prematurement les essais peu prometteurs (`pruning`).
4.  **Visualisation des Resultats :** Optuna fournit des outils de visualisation puissants pour analyser l'etude d'optimisation, comprendre l'importance des hyperparametres et observer les interactions.
5.  **Application sur un Modele Puissant :** Nous utilisons un classifieur `XGBoost`, un modele tres performant mais qui possede de nombreux hyperparametres a regler.

_Derniere mise a jour : 2026-02-16_

In [ ]:
# --- 1. Installation des Dependances ---
# Optuna est utilise pour l'optimisation, plotly pour les visualisations interactives.
%pip install -q optuna xgboost scikit-learn plotly
print("Dependances installees.")

In [ ]:
# --- 2. Imports ---
import optuna
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
import logging

# --- Configuration ---
optuna.logging.set_verbosity(optuna.logging.WARNING) # Reduit la verbosite d'Optuna
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 3. Preparation des Donnees

Nous utilisons le jeu de donnees `breast_cancer` de scikit-learn. C'est un probleme de classification binaire simple, ideal pour se concentrer sur le processus d'optimisation.

In [ ]:
# Chargement et preparation des donnees
logger.info("Chargement du jeu de donnees breast_cancer...")
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

logger.info(f"Taille de l'ensemble d'entrainement: {X_train.shape[0]} echantillons")
logger.info(f"Taille de l'ensemble de test: {X_test.shape[0]} echantillons")

## 4. Definition de la Fonction Objective

C'est le cœur du processus avec Optuna. La fonction `objective` recoit un objet `trial` et l'utilise pour :
1.  **Suggérer des hyperparametres :** `trial.suggest_float`, `trial.suggest_int`, etc., definissent l'espace de recherche.
2.  **Creer et entrainer un modele** avec ces hyperparametres.
3.  **Evaluer le modele** (ici avec une validation croisee a 3-folds pour la robustesse).
4.  **Retourner un score** (la precision moyenne) qu'Optuna tentera de maximiser.

In [ ]:
def objective(trial):
    """Fonction objective qu'Optuna cherchera a maximiser."""
    
    # Definition de l'espace de recherche des hyperparametres
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'lambda': trial.suggest_float('lambda', 1, 10),
        'alpha': trial.suggest_float('alpha', 0, 5),
    }

    # Creation du modele avec les hyperparametres suggeres
    model = xgb.XGBClassifier(**param, random_state=42)
    
    # Evaluation robuste avec validation croisee
    score = cross_val_score(model, X_train, y_train, n_jobs=-1, cv=3, scoring='accuracy')
    accuracy = score.mean()
    
    return accuracy

## 5. Lancement de l'Etude d'Optimisation

Nous creons une `study` Optuna, en lui indiquant que nous voulons maximiser le score. `study.optimize` lance le processus pour un nombre defini d'essais (`n_trials`).

In [ ]:
logger.info("Lancement de l'etude d'optimisation Optuna...")

# Creer une etude. Le `sampler` TPE est l'algorithme par defaut et est tres performant.
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

# Lancer l'optimisation
study.optimize(objective, n_trials=50, n_jobs=-1) # 50 essais, en parallele

logger.info("Optimisation terminee.")

# Affichage des resultats
print(f"Nombre d'essais termines: {len(study.trials)}")
print("Meilleur essai:")
best_trial = study.best_trial

print(f"  Valeur (Precision): {best_trial.value:.4f}")
print("  Meilleurs Hyperparametres:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

## 6. Analyse Visuelle des Resultats

Optuna excelle dans la visualisation, ce qui est crucial pour comprendre la recherche. Si `plotly` est installe, les graphiques suivants seront interactifs.

- **`plot_optimization_history`**: Montre la progression de la meilleure precision trouvee au fil des essais.
- **`plot_param_importances`**: Indique quels hyperparametres ont eu le plus d'impact sur la performance.
- **`plot_slice`**: Permet d'isoler un hyperparametre et de voir son influence sur le score.

In [ ]:
# Ces fonctions ne fonctionneront que dans un environnement ou plotly peut s'afficher.
try:
    # Historique de l'optimisation
    fig_history = optuna.visualization.plot_optimization_history(study)
    fig_history.update_layout(title_text="Historique de l Optimisation")
    fig_history.show()
    
    # Importance des hyperparametres
    fig_importance = optuna.visualization.plot_param_importances(study)
    fig_importance.update_layout(title_text="Importance des Hyperparametres")
    fig_importance.show()

    # Graphique 'slice' pour voir l'impact de chaque hyperparametre
    fig_slice = optuna.visualization.plot_slice(study)
    fig_slice.update_layout(title_text="Impact Individuel des Hyperparametres (Slice Plot)")
    fig_slice.show()
    
except (ImportError, ValueError):
    logger.warning("Plotly n est pas entierement configure. Ignore les visualisations interactives.")

## 7. Entrainement du Modele Final

Maintenant que nous avons les meilleurs hyperparametres, nous pouvons entrainer un modele final sur l'ensemble des donnees d'entrainement et l'evaluer sur l'ensemble de test que nous avions mis de cote.

In [ ]:
# Recuperer les meilleurs hyperparametres
best_params = best_trial.params

# Creer et entrainer le modele final
logger.info("Entrainement du modele final avec les meilleurs hyperparametres...")
final_model = xgb.XGBClassifier(**best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluation finale sur l'ensemble de test
y_pred = final_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

logger.info(f"Precision finale sur l'ensemble de test: {final_accuracy:.4f}")

In [ ]:
# Marqueur d'execution pour garantir au moins une sortie
print('Notebook execute avec succes — ' + time.strftime('%Y-%m-%d %H:%M:%S'))